# Notebook 1: Setup & Materialize on OpenShift AI

This notebook walks through the one-time setup:
1. Install Feast SDK and dependencies
2. Clone the demo repo and generate sample data
3. Apply Feast feature definitions
4. Materialize features to PostgreSQL + pgvector online store

**Prerequisites:** This workbench should be running in the `life-insurance-rag` namespace
with PostgreSQL and the Feast FeatureStore CR already deployed.

In [ ]:
# Install Feast SDK and dependencies in the workbench
!pip install -q 'feast[postgres]' sentence-transformers numpy pandas psycopg

In [ ]:
# Clone the demo repo (skip if already cloned)
import os
REPO_DIR = os.path.expanduser('~/life-insurance-rag')
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/ntkathole/life-insurance-rag.git {REPO_DIR}
    print(f'Cloned to {REPO_DIR}')
else:
    print(f'Repo already exists at {REPO_DIR}')
os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# Verify PostgreSQL connectivity
import os
pg_host = os.environ.get('FEAST_PG_HOST', 'postgres.life-insurance-rag.svc.cluster.local')
pg_port = os.environ.get('FEAST_PG_PORT', '5432')
print(f'PostgreSQL: {pg_host}:{pg_port}')

import psycopg
conn = psycopg.connect(
    host=pg_host, port=pg_port,
    dbname='feast', user='feast', password='feast'
)
cur = conn.execute('SELECT version()')
print(f'Connected: {cur.fetchone()[0]}')
cur = conn.execute("SELECT * FROM pg_extension WHERE extname='vector'")
row = cur.fetchone()
if row:
    print(f'pgvector extension: installed')
else:
    conn.execute('CREATE EXTENSION IF NOT EXISTS vector')
    conn.commit()
    print('pgvector extension: just installed')
conn.close()

## Step 1: Generate Sample Data

In [ ]:
!python data/generate_sample_data.py

In [ ]:
import pandas as pd

profiles = pd.read_parquet('feature_repo/data/customer_profiles.parquet')
docs = pd.read_parquet('feature_repo/data/insurance_docs.parquet')
claims = pd.read_parquet('feature_repo/data/claim_status.parquet')

print(f'Customers: {len(profiles)}')
print(f'Documents: {len(docs)}')
print(f'Active claims: {len(claims)}')
profiles.head()

In [ ]:
docs[['doc_id', 'title', 'category', 'plan_type']].head(15)

## Step 2: Write feature_store.yaml for this environment

We create a local `feature_store.yaml` that points to the PostgreSQL instance in the cluster.
This is needed for `feast apply` and the materialize script.

In [ ]:
import os

pg_host = os.environ.get('FEAST_PG_HOST', 'postgres.life-insurance-rag.svc.cluster.local')
pg_port = os.environ.get('FEAST_PG_PORT', '5432')
pg_user = os.environ.get('POSTGRES_USER', 'feast')
pg_pass = os.environ.get('POSTGRES_PASSWORD', 'feast')
pg_db   = os.environ.get('POSTGRES_DB', 'feast')

feature_store_yaml = f"""project: life_insurance_rag
provider: local

registry:
  registry_type: sql
  path: postgresql+psycopg://{pg_user}:{pg_pass}@{pg_host}:{pg_port}/{pg_db}
  cache_ttl_seconds: 60

online_store:
  type: postgres
  host: {pg_host}
  port: {pg_port}
  database: {pg_db}
  db_schema: public
  user: {pg_user}
  password: {pg_pass}
  sslmode: disable
  vector_enabled: true

offline_store:
  type: file

embedding_model:
  provider: sentence_transformers
  model: all-MiniLM-L6-v2

auth:
  type: no_auth

entity_key_serialization_version: 3
"""

with open('feature_repo/feature_store.yaml', 'w') as f:
    f.write(feature_store_yaml)

print('Wrote feature_repo/feature_store.yaml')
print(feature_store_yaml)

## Step 3: Apply Feast Definitions

Registers entities, feature views, and on-demand feature views with the SQL registry.

In [ ]:
!cd feature_repo && feast apply

## Step 4: Materialize to Online Store

Pushes all feature data (documents, profiles, claims) into PostgreSQL.

In [ ]:
!python scripts/materialize.py

## Verify: Check the Online Store

In [ ]:
from feast import FeatureStore

store = FeatureStore(repo_path='feature_repo')

result = store.get_online_features(
    features=[
        'customer_profile:customer_name',
        'customer_profile:plan_type',
        'customer_profile:coverage_amount',
    ],
    entity_rows=[{'customer_id': 'POL-1001'}],
).to_dict()

print('Customer lookup result:')
for k, v in result.items():
    print(f'  {k}: {v}')

## Verify: Check the FeatureStore CR services (operator-managed)

The Feast operator automatically deploys registry and online store servers.
Check their endpoints:

In [ ]:
import os
feast_url = os.environ.get('FEAST_SERVER_URL', 'http://feast-life-insurance-rag-online.life-insurance-rag.svc.cluster.local:80')
print(f'Feast online server: {feast_url}')

# Quick health check
try:
    import urllib.request
    resp = urllib.request.urlopen(f'{feast_url}/health', timeout=5)
    print(f'Health check: {resp.status} {resp.read().decode()}')
except Exception as e:
    print(f'Health check: {e}')
    print('(The operator-managed server may still be starting up)')

---
**Setup complete!** Open `02_level0_basic_rag.ipynb` to start the RAG demo.